# **Prédire le contenu d'un panier pour un acheteur d'alcool**

## Import

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, KFold
from scipy.stats import randint, uniform

## Modèle

In [3]:
df = pd.read_csv("./datasets-cleaned/df_order_products_clean.csv")

# Récupère si l'article dans le panier est de l'alcool
df["is_alcohol"] = df["department"].str.contains("alcohol", case=False, na=False).astype(int)

# Regroupe toutes les lignes qui on le même "order_id" ensuite compte le nombre de lignes dans chaque groupe ( donc le nombre de produit ) et transforme le résultat en dataframe
basket_size = df.groupby("order_id").size().reset_index(name="basket_size")

orders = df.groupby("order_id").agg(
    user_id = ("user_id", "first"),
    order_number = ("order_number", "first"),
    order_hour_of_day = ("order_hour_of_day", "first"),
    order_dow = ("order_dow", "first"), 
    days_since_prior_order = ("days_since_prior_order", "first"),
    alcohol_count = ("is_alcohol", "sum"),
    reorder_count = ("reordered", "sum"),
    number_different_departments = ("department", "nunique"),
    number_different_aisles = ("aisle", "nunique"),
).reset_index()

orders = orders.merge(basket_size, on="order_id")

# Ici on récupère si c'est le weekend ( classique burger ) 
orders["is_weekend"] = orders["order_dow"].isin([0, 6]).astype(int)
# Ici on récupère le moment de la journée (oui j'ai remis le 24 parceque sinon ça prenait pas en compte 22 et 23h singe) 
orders["time_of_day"] = pd.cut(
    orders["order_hour_of_day"],
    bins=[0, 6, 12, 17, 21, 24],
    labels=["nuit", "matin", "apres_midi", "soir", "soiree"],
    right=False
).astype(str)

# On fait le tri par utilisateur et numéro de commande
orders_sorted = orders.sort_values(["user_id", "order_number"])

# Ça en résumé : Je fais une moyenne ligne par ligne et à chaque ligne avec le ".expanding" je fais la moyenne en récupérant la ligne d'avant aussi ex : commande 1 = 5 la moyenne c'est 5, commande 2 = [8 ET 5] donc la moyenne c'est 6.5 etc ...
orders_sorted["user_avg_basket"] = (
    orders_sorted.groupby("user_id")["basket_size"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

# Taille du panier précedent
orders_sorted["user_last_basket"] = (
    orders_sorted.groupby("user_id")["basket_size"]
    .transform(lambda x: x.shift(1))
)

# Nombre de commandes d'avant
orders_sorted["user_order_count"] = orders_sorted["order_number"] - 1

# Moyenne de rachat d'avant
orders_sorted["user_avg_reorder"] = (
    orders_sorted.groupby("user_id")["reorder_count"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

# Remplacer les Na par la médiane globale
for col in ["user_avg_basket", "user_last_basket", "user_avg_reorder"]:
    orders_sorted[col] = orders_sorted[col].fillna(orders_sorted[col].median())

orders = orders_sorted.copy()

#/--------------------------------------------------------------\#
#|------------------------ Entraînement ------------------------|#
#\--------------------------------------------------------------/#

feature_cols = [
    "order_hour_of_day", "order_dow",
    "is_weekend", "time_of_day",
    "days_since_prior_order", "order_number",
    "alcohol_count", "reorder_count",
    "number_different_departments", "number_different_aisles",
    "user_avg_basket", "user_last_basket",
    "user_order_count", "user_avg_reorder",
]

X = pd.get_dummies(orders[feature_cols])
y = orders["basket_size"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.05, random_state=42
)

model = HistGradientBoostingRegressor(
    max_iter=440,
    learning_rate=np.float64(0.026183714048995665),
    max_leaf_nodes=82,
    min_samples_leaf=13,
    l2_regularization=np.float64(0.012709563372047594),
    random_state=150,
    max_depth=9
)
# Lance l'entraînements du modèle
model.fit(X_train, y_train)

#/--------------------------------------------------------------\#
#|------------------------- Évaluation -------------------------|#
#\--------------------------------------------------------------/#

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"MAE :{mae}")
print(f"RMSE : {rmse}")
print(f"R2 : {r2}")

#/---------------------------------------------------------------\#
#|------------------------ Justification ------------------------|#
#\---------------------------------------------------------------/#

# param_dist = {
#     "max_iter": randint(200, 800),
#     "learning_rate": uniform(0.01, 0.15),
#     "max_leaf_nodes": randint(31, 512),
#     "min_samples_leaf": randint(10, 100),
#     "l2_regularization": uniform(0.0, 0.5),
#     "max_depth": randint(3, 12),
#     "random_state": randint(0, 200)
# }

# cv = KFold(n_splits=5, shuffle=True, random_state=42)

# search = RandomizedSearchCV(
#     estimator = HistGradientBoostingRegressor(),
#     param_distributions = param_dist,
#     n_iter = 30,
#     scoring = "r2",
#     cv = cv,
#     n_jobs = -1,
#     verbose = 2,
#     random_state= 42,
#     return_train_score = True
# )

# search.fit(X_train, y_train)

# print(search.best_params_)
# print(f"Meilleur R² (CV) : {search.best_score_:.4f}")

# best_model = search.best_estimator_
# y_pred = best_model.predict(X_test)
# r2_2 = r2_score(y_test, y_pred)
# print(f"R² final (test) : {r2_2}")

MAE :1.3271381401047326
RMSE : 4.131955381317046
R2 : 0.9256455171714738
